In [2]:
import pandas as pd
t=pd.read_csv("amazon_reviews.csv")
y=pd.read_csv("yelp_reviews.csv")
u=pd.read_csv("fake_reviews_dataset[1].csv")

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import joblib

# 1. Load your dataset
print("Loading data...")
df = pd.read_csv('fake_reviews_dataset[1].csv')

# 2. Setup the Pipeline (Text Processing + Classifier)
# We use TF-IDF to turn text into numbers and Naive Bayes to classify them
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_df=0.7)),
    ('classifier', MultinomialNB())
])

# 3. Split data (80% for training, 20% for testing)
X = df['text_']
y = df['label'] # CG = Fake, OR = Human
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the model
print("Training the model (this may take a few seconds)...")
model.fit(X_train, y_train)

# 5. Save the trained model to a file
joblib.dump(model, 'review_detector.bin')
print("Success! Your model is saved as 'review_detector.bin'")

Loading data...
Training the model (this may take a few seconds)...
Success! Your model is saved as 'review_detector.bin'


In [5]:
import joblib

# Load your newly trained model
model = joblib.load('review_detector.bin')

# Enter any review text here
test_reviews = [
    "I love this product! It works perfectly and arrived on time.", # Likely Human (OR)
    "This is a very good product. I am happy with the purchase. Good quality." # Likely AI (CG)
]

# Get predictions
predictions = model.predict(test_reviews)

for text, label in zip(test_reviews, predictions):
    result = "Human (Real)" if label == 'OR' else "AI (Fake)"
    print(f"Review: {text[:50]}... \nResult: {result}\n")

Review: I love this product! It works perfectly and arrive... 
Result: AI (Fake)

Review: This is a very good product. I am happy with the p... 
Result: AI (Fake)



In [6]:
import pandas as pd
import joblib

# 1. Load the trained model
try:
    model = joblib.load('review_detector.bin')
    print("Model loaded successfully!\n")
except:
    print("Error: 'review_detector.bin' not found. Run the training script first.")
    exit()

def test_file(file_name, text_column):
    print(f"--- Analyzing {file_name} ---")
    
    # Load data
    df = pd.read_csv(file_name)
    
    # Remove empty reviews to avoid errors
    df = df.dropna(subset=[text_column])
    
    # Predict
    # CG = Computer Generated (Fake), OR = Original (Human)
    predictions = model.predict(df[text_column])
    
    # Add predictions back to the dataframe for viewing
    df['predicted_label'] = predictions
    df['result'] = df['predicted_label'].map({'CG': 'AI/Fake', 'OR': 'Human/Real'})
    
    # Show summary
    counts = df['result'].value_counts()
    print("Results Summary:")
    print(counts)
    
    # Show a few examples of what it flagged as AI
    ai_examples = df[df['predicted_label'] == 'CG'][text_column].head(2)
    if not ai_examples.empty:
        print("\nExample(s) flagged as AI/Fake:")
        for ex in ai_examples:
            print(f"- {ex[:100]}...")
    
    print("-" * 30 + "\n")
    return df

# 2. Run test on Yelp (Column name is 'Review')
yelp_results = test_file('yelp_reviews.csv', 'Review')

# 3. Run test on Amazon (Column name is 'verified_reviews')
amazon_results = test_file('amazon_reviews.csv', 'verified_reviews')

# Optional: Save the results to new CSVs to inspect in VS Code
yelp_results.to_csv('yelp_with_predictions.csv', index=False)
amazon_results.to_csv('amazon_with_predictions.csv', index=False)
print("Saved labeled results to 'yelp_with_predictions.csv' and 'amazon_with_predictions.csv'")

Model loaded successfully!

--- Analyzing yelp_reviews.csv ---
Results Summary:
result
Human/Real    47
AI/Fake        3
Name: count, dtype: int64

Example(s) flagged as AI/Fake:
- The line is totally worth it! The moment you walk in, you can smell the fresh made waffles!

Definit...
- Nice ice cream shop. Order at the counter and then there's a counter inside to eat or tables out.

S...
------------------------------

--- Analyzing amazon_reviews.csv ---
Results Summary:
result
Human/Real    2026
AI/Fake       1123
Name: count, dtype: int64

Example(s) flagged as AI/Fake:
- Loved it!...
- I have had a lot of fun with this thing. My 4 yr old learns about dinosaurs, i control the lights an...
------------------------------

Saved labeled results to 'yelp_with_predictions.csv' and 'amazon_with_predictions.csv'


In [12]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the model
try:
    model = joblib.load('review_detector.bin')
except:
    print("Please run your training script first to create 'review_detector.bin'")
    exit()

# 2. Load the labeled data
df = pd.read_csv('fake_reviews_dataset[1].csv')
df = df.dropna(subset=['text_'])

# 3. Get Predictions
print("Calculating accuracy...")
X = df['text_']
y_true = df['label'] # The actual answer (CG or OR)
y_pred = model.predict(X) # What the model thinks

# 4. Calculate Scores
acc = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['AI (CG)', 'Human (OR)'])

print(f"\nOVERALL ACCURACY: {acc * 100:.2f}%")
print("\nDETAILED REPORT:")
print(report)

# 5. Confusion Matrix (See where the model gets confused)
cm = confusion_matrix(y_true, y_pred)
print("\nCONFUSION MATRIX:")
print(f"True Human identified as Human: {cm[1][1]}")
print(f"True Human identified as AI:    {cm[1][0]} (Errors)")
print(f"True AI identified as AI:       {cm[0][0]}")
print(f"True AI identified as Human:    {cm[0][1]} (Errors)")

Calculating accuracy...

OVERALL ACCURACY: 87.37%

DETAILED REPORT:
              precision    recall  f1-score   support

     AI (CG)       0.83      0.94      0.88     20216
  Human (OR)       0.93      0.81      0.86     20216

    accuracy                           0.87     40432
   macro avg       0.88      0.87      0.87     40432
weighted avg       0.88      0.87      0.87     40432


CONFUSION MATRIX:
True Human identified as Human: 16361
True Human identified as AI:    3855 (Errors)
True AI identified as AI:       18964
True AI identified as Human:    1252 (Errors)


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import joblib

# 1. Load data
print("Loading dataset...")
df = pd.read_csv('fake_reviews_dataset[1].csv')
df = df.dropna(subset=['text_'])

# 2. Setup the PRO Pipeline
# - ngram_range=(1, 2) lets it see word pairs
# - max_features=20000 focuses on the most important phrases
# - LinearSVC is a high-performance classifier
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=20000)),
    ('classifier', LinearSVC(dual=False))
])

# 3. Split data
X = df['text_']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the model
print("Training the enhanced model (this might take a minute)...")
model.fit(X_train, y_train)

# 5. Check the new accuracy
accuracy = model.score(X_test, y_test)
print(f"\n--- SUCCESS! ---")
print(f"New Accuracy: {accuracy * 100:.2f}%")

# 6. Save the new version
joblib.dump(model, 'review_detector_v2.bin')
print("Saved as 'review_detector_v2.bin'")

Loading dataset...
Training the enhanced model (this might take a minute)...

--- SUCCESS! ---
New Accuracy: 93.88%
Saved as 'review_detector_v2.bin'


In [ ]:
import joblib
import time

# 1. Load the high-accuracy model
try:
    model = joblib.load('review_detector_v2.bin')
    print("--- Model Loaded & Ready! ---")
    print("Type a review to check if it's AI or Human.")
    print("Type 'exit' to stop.\n")
except:
    print("Error: 'review_detector_v2.bin' not found. Run your training script first!")
    exit()

# 2. Start the Input Loop
while True:
    user_review = input("Enter Review: ").strip()
    
    if user_review.lower() == 'exit':
        print("Closing tester...")
        break
    
    if not user_review:
        continue

    # 3. Make Prediction
    # [Text] needs to be in a list because the model expects a batch
    start_time = time.time()
    prediction = model.predict([user_review])[0]
    end_time = time.time()

    # 4. Interpret Result
    # CG = Computer Generated (Fake), OR = Original (Human)
    if prediction == 'CG':
        result = "🚩 AI-GENERATED (FAKE)"
        color = "\033[91m" # Red text in terminal
    else:
        result = "✅ HUMAN (ORIGINAL)"
        color = "\033[92m" # Green text in terminal

    # Reset color
    reset = "\033[0m"

    print(f"Result: {color}{result}{reset}")
    print(f"Confidence check took: {end_time - start_time:.4f} seconds\n")

--- Model Loaded & Ready! ---
Type a review to check if it's AI or Human.
Type 'exit' to stop.



Enter Review:  If you have pets or hardwood floors, this is worth the extra few dollars.


Result: 🚩 AI-GENERATED (FAKE)
Confidence check took: 0.0023 seconds

